In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set random seed for reproducibility
np.random.seed(42)

class Perceptron:
    """Single-layer Perceptron Classifier using step activation."""
    def __init__(self, input_dim: int, lr: float = 0.1):
        self.weights = np.random.randn(input_dim + 1)  # Includes bias weight
        self.lr = lr
        self.history = []  # Stores weight snapshots after every update

    def predict(self, X: np.ndarray) -> np.ndarray:
        # X: (N, D) -> Add column of ones for bias
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        linear_output = X_b @ self.weights
        return (linear_output >= 0).astype(int)

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 20):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.history = [self.weights.copy()]
        
        for epoch in range(epochs):
            errors_in_epoch = 0
            for i in range(len(y)):
                prediction = (X_b[i] @ self.weights >= 0).astype(int)
                error = y[i] - prediction
                if error != 0:
                    self.weights += self.lr * error * X_b[i]
                    self.history.append(self.weights.copy())
                    errors_in_epoch += 1
            if errors_in_epoch == 0:
                break
        return self

In [ ]:
def plot_logic_gate(X: np.ndarray, y: np.ndarray, title: str):
    """Visualizes 2D binary classification datasets."""
    plt.figure(figsize=(5, 5))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color='red', s=100, label='0', edgecolors='k')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='blue', s=100, label='1', edgecolors='k')
    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()

def animate_decision_boundary(model: Perceptron, X: np.ndarray, y: np.ndarray, title: str):
    """Generates an animated decision boundary tracking perceptron learning steps."""
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Scatter plot data points
    ax.scatter(X[y == 0, 0], X[y == 0, 1], color='red', s=100, label='0', edgecolors='k')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color='blue', s=100, label='1', edgecolors='k')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
    line, = ax.plot([], [], 'g--', linewidth=2, label='Decision Boundary')

    def init():
        line.set_data([], [])
        return line,

    def update(frame):
        w0, w1, w2 = model.history[frame]
        # Line equation: w0 + w1*x1 + w2*x2 = 0 => x2 = -(w1*x1 + w0) / w2
        x_vals = np.array([-0.5, 1.5])
        if abs(w2) > 1e-5:
            y_vals = -(w1 * x_vals + w0) / w2
        else:
            y_vals = np.array([-0.5, 1.5])
            
        line.set_data(x_vals, y_vals)
        ax.set_title(f"{title} - Step {frame}/{len(model.history)-1}")
        return line,

    anim = FuncAnimation(fig, update, frames=len(model.history), init_func=init, blit=True, interval=400)
    plt.close()
    return HTML(anim.to_jshtml())

In [ ]:
# Datasets for Logic Gates
X_gate = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

# AND Gate Target Labels
y_and = np.array([0, 0, 0, 1])

# Train Perceptron on AND Gate
and_perceptron = Perceptron(input_dim=2, lr=0.1)
and_perceptron.fit(X_gate, y_and, epochs=20)

print(f"AND Gate Trained in {len(and_perceptron.history)-1} weight updates.")
animate_decision_boundary(and_perceptron, X_gate, y_and, "AND Gate Learning Trajectory")

In [ ]:
# OR Gate Target Labels
y_or = np.array([0, 1, 1, 1])

# Train Perceptron on OR Gate
or_perceptron = Perceptron(input_dim=2, lr=0.1)
or_perceptron.fit(X_gate, y_or, epochs=20)

print(f"OR Gate Trained in {len(or_perceptron.history)-1} weight updates.")
animate_decision_boundary(or_perceptron, X_gate, y_or, "OR Gate Learning Trajectory")

In [ ]:
# XOR Gate Target Labels
y_xor = np.array([0, 1, 1, 0])

# Train Perceptron on XOR Gate
xor_perceptron = Perceptron(input_dim=2, lr=0.1)
xor_perceptron.fit(X_gate, y_xor, epochs=50)

print("Displaying XOR Dataset:")
plot_logic_gate(X_gate, y_xor, "XOR Problem (Non-Linearly Separable)")

print("Attempting to animate XOR convergence...")
animate_decision_boundary(xor_perceptron, X_gate, y_xor, "XOR Failure Trajectory")